# Experiment 2.2c: Self-Correction Timescale -- Steps to Halve Momentum Imbalance vs Rescaling Factor $c$

---

## Theoretical Background

In deep linear networks, there exists a continuous family of **reparametrization symmetries**:
for any invertible matrix $C$, one can replace $(W_{\ell+1}, W_\ell)$ with
$(W_{\ell+1} C^{-1}, C W_\ell)$ without changing the network's input-output map.
A scalar version of this is **layerwise rescaling**: multiply layer 0 by $c$ and
layer $L-1$ by $1/c$, preserving the product $W_L \cdots W_1$.

Experiment 2.2b revealed a striking surprise: **Muon survives rescaling factors as
large as $c = 1000$** where SGD diverges to NaN. The mechanism is self-correcting:
the momentum imbalance ratio (max/min momentum norm across layers) starts at
$\sim 9 \times 10^5$ but decays to $\sim 296$ over 300 steps. The Newton-Schulz
step's per-layer Frobenius normalization absorbs extreme scales.

## The Key Question

This experiment quantifies the **timescale** of this self-correction:

1. **Half-life**: How many steps until the momentum imbalance drops to 50% of its initial value?
2. **Scaling law**: Does the half-life scale as $O(\log c)$, suggesting logarithmically efficient correction?
3. **Learning vs. rebalancing**: Does the network start learning (loss decreasing) *before* or *after* the momentum rebalances?

The third question is especially important: if learning begins before rebalancing completes,
it means Muon's gauge-fixing operates concurrently with useful optimization, not as a
prerequisite phase.

## Physical Analogy

This is analogous to measuring the **relaxation time** of a physical system after a
perturbation. In RG language, the rescaling $c$ introduces an "irrelevant operator"
that should flow to zero under the RG flow. The half-life measures how quickly this
irrelevant perturbation decays. $O(\log c)$ scaling would indicate exponential decay,
characteristic of a system near a fixed point with a finite gap in the linearized
RG operator.

## Experimental Design

| Parameter | Value | Rationale |
|:----------|:------|:----------|
| Rescaling values $c$ | {1, 10, 100, 1000, 10000} | Spans 4 orders of magnitude |
| Network depth | 4 layers | Minimal depth for non-trivial layer effects |
| Matrix dimension | 32x32 | Large enough for meaningful spectra |
| Training steps | 500 | Long enough to observe full correction dynamics |
| Seeds | 5 | Statistical robustness |

## Key Tests

| Test | Criterion | Physical meaning |
|:-----|:----------|:-----------------|
| T1 | Half-life slope < 50 steps/decade of $c$ | Self-correction is efficient (sub-linear in $c$) |
| T2 | Half-life < 50 steps for $c \le 1000$ | Correction is fast in the practical regime |
| T3 | Training onset < half-life | Learning and correction are concurrent |

## Environment Setup

Import NumPy for linear algebra and matrix operations. This experiment uses only
CPU-based computation on small 32x32 matrices.

In [ ]:
import numpy as np
import os

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath('.'))


## Experimental Configuration

Define the hyperparameters governing the experiment.

**Key design choices**:
- 500 training steps (longer than the standard 300) to ensure we observe full correction dynamics
  even for extreme $c$ values
- Momentum $\beta = 0.9$ as in standard Muon; momentum buffers are the primary vehicle
  through which scale imbalance propagates
- EMA-style momentum update: $m_t = \beta\,m_{t-1} + (1-\beta)\,g_t$, which means the
  momentum buffer tracks a running average of gradients (vs. classical momentum which
  accumulates)

In [ ]:
DIM = 32
N_LAYERS = 4
NUM_STEPS = 500
LR = 0.02
MOMENTUM_BETA = 0.9
NS_ITERS = 5
NUM_SEEDS = 5
BATCH_SIZE = 128

print("\n--- Experimental Configuration ---")
print(f"  DIM = {DIM}")
print(f"  NUM_STEPS = {NUM_STEPS}")
print(f"  BATCH_SIZE = {BATCH_SIZE}")
print(f"  NUM_SEEDS = {NUM_SEEDS}")
print(f"  LR = {LR}")
print(f"  NS_ITERS = {NS_ITERS}")
print(f"  N_LAYERS = {N_LAYERS}")


The rescaling values span 4 orders of magnitude. At $c=1$, no rescaling occurs
(baseline). At $c=10000$, layer 0 has norms $10^4$ times larger than normal while
layer $L-1$ has norms $10^4$ times smaller -- an extreme stress test of scale invariance.

In [ ]:
C_VALUES = [1, 10, 100, 1000, 10000]

## Core Algorithms

### Newton-Schulz Orthogonalization

The Newton-Schulz iteration $X_{k+1} = \frac{3}{2}X_k - \frac{1}{2}X_k(X_k^\top X_k)$
converges to the nearest orthogonal matrix. The crucial first step is **Frobenius
normalization**: $X_0 = M / \|M\|_F$. This normalization is what absorbs extreme scales --
a momentum buffer with $\|M\|_F = 10^6$ and one with $\|M\|_F = 1$ both get mapped
to unit-Frobenius-norm before iteration begins, effectively erasing the scale information.

### Weight Initialization with Rescaling

Weights are initialized as $W_\ell \sim \mathcal{N}(0, 1/\sqrt{d})$ (Kaiming-like),
then layer 0 is multiplied by $c$ and layer $L-1$ is divided by $c$. This preserves
the end-to-end product $W_L \cdots W_1$ but creates a massive imbalance in per-layer norms.

In [ ]:
def newton_schulz(M, n_iters=NS_ITERS):
    norm = np.linalg.norm(M, ord='fro')
    if norm < 1e-15:
        return M
    X = M / norm
    for _ in range(n_iters):
        A = X.T @ X
        X = 1.5 * X - 0.5 * X @ A
    return X

In [ ]:
def init_weights(rng, c):
    weights = []
    for l in range(N_LAYERS):
        W = rng.randn(DIM, DIM) / np.sqrt(DIM)
        weights.append(W)
    weights[0] = weights[0] * c
    weights[-1] = weights[-1] / c
    return weights

## Network Forward Pass

The forward pass computes $\hat{Y} = W_L \cdots W_2 W_1 X$ -- a simple chain of
matrix multiplications. Despite the rescaling, the product $W_L \cdots W_1$ is
unchanged from the un-rescaled case, so the initial loss is identical regardless
of $c$. However, the gradients $\nabla_{W_\ell} \mathcal{L}$ scale differently
across layers due to the chain rule, creating the initial momentum imbalance.

In [ ]:
def forward(weights, X):
    out = X
    for W in weights:
        out = W @ out
    return out

## Gradient Computation

Standard backpropagation through the linear chain. The gradient at layer $\ell$ is
$\nabla_{W_\ell} \mathcal{L} = \delta_\ell \, a_{\ell-1}^\top$ where $\delta_\ell$
is the backpropagated error. Crucially, the rescaling by $c$ means:
- Layer 0 has activations scaled by $c$, so $\|\nabla_{W_0}\|$ is amplified
- Layer $L-1$ has weights scaled by $1/c$, affecting the backprop path

This asymmetry creates the initial momentum imbalance that Muon must self-correct.

In [ ]:
def compute_gradients(weights, X, Y):
    L = len(weights)
    N = X.shape[1]
    acts = [X]
    for W in weights:
        acts.append(W @ acts[-1])
    delta = 2.0 * (acts[-1] - Y) / N
    grads = [None] * L
    for l in range(L - 1, -1, -1):
        grads[l] = delta @ acts[l].T
        if l > 0:
            delta = weights[l].T @ delta
    return grads

## Loss Function

MSE loss $\mathcal{L} = \frac{1}{N}\sum_j \|\hat{y}_j - y_j\|^2$. We track the loss
at every step to determine the **training onset** -- the first step where the loss drops
below 99% of its initial value. This onset time is compared to the momentum half-life
to determine whether learning precedes or follows rebalancing.

In [ ]:
def compute_loss(weights, X, Y):
    pred = forward(weights, X)
    return np.mean((pred - Y) ** 2)

## Training Loop with Momentum Imbalance Tracking

The training function runs Muon optimization while recording two time series at every step:

1. **Momentum imbalance**: $\text{imbalance}(t) = \frac{\max_\ell \|m_\ell(t)\|_F}{\min_\ell \|m_\ell(t)\|_F}$
   
   This ratio starts at $\sim c^2$ (since the rescaling creates $O(c^2)$ differences in
   gradient norms between the scaled and un-scaled layers) and should decay as Newton-Schulz
   normalization progressively absorbs the scale imbalance.

2. **Loss history**: $\mathcal{L}(t)$ at every step, used to detect training onset.

Note that this uses EMA-style momentum $(1-\beta)g + \beta m$, not classical momentum
$g + \beta m$. The $(1-\beta)$ factor slightly dampens new gradient contributions,
which may slow initial correction but provides more stable long-term dynamics.

In [ ]:
def train_and_track(weights_init, X, Y):
    weights = [W.copy() for W in weights_init]
    mom = [np.zeros_like(W) for W in weights]
    imbalance_history = []
    loss_history = []

    for step in range(NUM_STEPS):
        loss = compute_loss(weights, X, Y)
        loss_history.append(loss)
        if not np.isfinite(loss) or loss > 1e10:
            break

        grads = compute_gradients(weights, X, Y)
        for l in range(N_LAYERS):
            mom[l] = MOMENTUM_BETA * mom[l] + (1 - MOMENTUM_BETA) * grads[l]
            ortho_mom = newton_schulz(mom[l])
            weights[l] = weights[l] - LR * ortho_mom

        # Track momentum imbalance
        mom_norms = [np.linalg.norm(m, 'fro') for m in mom]
        imbalance = max(mom_norms) / max(min(mom_norms), 1e-30)
        imbalance_history.append(imbalance)

    return imbalance_history, loss_history

## Timescale Measurement Functions

### Half-Life Detection

The half-life is the first training step $t^*$ such that:
$$\text{imbalance}(t^*) \le 0.5 \times \text{imbalance}(0)$$

This is a standard relaxation measure. If the imbalance decays exponentially
as $\text{imbalance}(t) \approx \text{imbalance}(0) \cdot e^{-t/\tau}$, then
$t^* = \tau \ln 2 \approx 0.693\,\tau$.

### Training Onset Detection

The training onset is the first step where the loss drops below 99% of the initial
loss. This threshold is generous -- even a 1% decrease indicates the optimizer has
found a useful gradient direction despite the scale imbalance.

In [ ]:
def find_half_life(imbalance_history):
    """Find first step where imbalance drops below 0.5 * initial."""
    if not imbalance_history:
        return float('nan')
    initial = imbalance_history[0]
    target = initial * 0.5
    for i, v in enumerate(imbalance_history):
        if v <= target:
            return i
    return float('nan')

In [ ]:
def find_training_onset(loss_history):
    """Find first step where loss decreases from initial."""
    if not loss_history:
        return float('nan')
    initial = loss_history[0]
    for i in range(1, len(loss_history)):
        if loss_history[i] < initial * 0.99:
            return i
    return float('nan')

## Main Experiment: Sweep Over Rescaling Factors

For each rescaling factor $c \in \{1, 10, 100, 1000, 10000\}$, we:
1. Initialize the rescaled network
2. Train with Muon, recording momentum imbalance and loss at every step
3. Measure the half-life and training onset
4. Average over 5 seeds for robustness

The key output is the relationship between $\log_{10}(c)$ and the half-life.
Linear scaling in $\log(c)$ would indicate that each additional order of magnitude
in rescaling adds a fixed number of correction steps -- a sign of exponential decay.

In [ ]:
def main():
    print("=" * 100)
    print("2.2c: SELF-CORRECTION TIMESCALE -- Steps to Halve Momentum Imbalance")
    print("=" * 100)
    print(f"c values: {C_VALUES}")
    print(f"Network: {N_LAYERS}-layer, {DIM}x{DIM}, {NUM_STEPS} steps, {NUM_SEEDS} seeds")
    print(f"LR = {LR}, Momentum beta = {MOMENTUM_BETA}, NS iterations = {NS_ITERS}")
    print()

    results = {}
    for c in C_VALUES:
        print(f"\n{'─' * 80}")
        print(f"  Rescaling factor c = {c} (log10(c) = {np.log10(c):.1f})")
        print(f"{'─' * 80}")

        half_lives = []
        onsets = []
        init_imbalances = []
        final_imbalances = []

        for seed_idx in range(NUM_SEEDS):
            rng = np.random.RandomState(42 + seed_idx * 137)
            X = rng.randn(DIM, BATCH_SIZE) * 0.3
            Y = rng.randn(DIM, BATCH_SIZE) * 0.3
            rng_w = np.random.RandomState(1000 + seed_idx)
            weights_init = init_weights(rng_w, c)

            # Report per-layer initial norms
            if seed_idx == 0:
                norms = [np.linalg.norm(W, 'fro') for W in weights_init]
                print(f"  Initial ||W||_F per layer: {['%.2e' % n for n in norms]}")
                print(f"  Norm ratio (max/min): {max(norms)/max(min(norms), 1e-30):.2e}")

            imb_hist, loss_hist = train_and_track(weights_init, X, Y)

            hl = find_half_life(imb_hist)
            onset = find_training_onset(loss_hist)
            half_lives.append(hl)
            onsets.append(onset)
            if imb_hist:
                init_imbalances.append(imb_hist[0])
                final_imbalances.append(imb_hist[-1])

            if seed_idx == 0:
                print(f"  Seed 0 details:")
                print(f"    Initial imbalance: {imb_hist[0]:.2e}" if imb_hist else "    No imbalance data")
                print(f"    Final imbalance:   {imb_hist[-1]:.2e}" if imb_hist else "    No imbalance data")
                print(f"    Half-life: {hl:.0f} steps" if np.isfinite(hl) else "    Half-life: not reached")
                print(f"    Training onset: step {onset:.0f}" if np.isfinite(onset) else "    Training onset: not reached")
                print(f"    Initial loss: {loss_hist[0]:.4e}" if loss_hist else "    No loss data")
                print(f"    Final loss:   {loss_hist[-1]:.4e}" if loss_hist else "    No loss data")
                # Show imbalance at a few checkpoints
                if imb_hist and len(imb_hist) > 10:
                    checkpoints = [0, 5, 10, 25, 50, 100, min(200, len(imb_hist)-1)]
                    print(f"    Imbalance trajectory: {['step %d: %.2e' % (t, imb_hist[t]) for t in checkpoints if t < len(imb_hist)]}")

        avg_hl = np.nanmean(half_lives)
        avg_onset = np.nanmean(onsets)
        print(f"  Averaged over {NUM_SEEDS} seeds:")
        print(f"    Mean half-life: {avg_hl:.1f} steps")
        print(f"    Mean onset: {avg_onset:.1f} steps")
        print(f"    Mean initial imbalance: {np.mean(init_imbalances):.2e}")
        print(f"    Mean final imbalance: {np.mean(final_imbalances):.2e}")
        print(f"    Compression ratio (init/final): {np.mean(init_imbalances)/max(np.mean(final_imbalances), 1e-30):.1f}x")

        results[c] = {
            'half_lives': half_lives,
            'onsets': onsets,
            'init_imbalance': np.mean(init_imbalances) if init_imbalances else float('nan'),
            'final_imbalance': np.mean(final_imbalances) if final_imbalances else float('nan'),
        }

    # Results table
    print(f"\n{'=' * 100}")
    print("CONSOLIDATED RESULTS")
    print(f"{'=' * 100}")

    print(f"\n  {'c':>8}  {'log10(c)':>8}  {'Init Imb':>12}  {'Final Imb':>12}  {'Half-life':>12}  {'Onset':>10}  {'HL < Onset?':>12}")
    print("  " + "-" * 80)

    for c in C_VALUES:
        r = results[c]
        hl_mean = np.nanmean(r['half_lives'])
        onset_mean = np.nanmean(r['onsets'])
        hl_before = hl_mean < onset_mean if np.isfinite(hl_mean) and np.isfinite(onset_mean) else False
        print(f"  {c:>8}  {np.log10(c):>8.1f}  {r['init_imbalance']:>12.2e}  {r['final_imbalance']:>12.2e}  "
              f"{hl_mean:>12.1f}  {onset_mean:>10.1f}  {'NO' if hl_before else 'YES':>12}")

    # Check O(log(c)) scaling
    print(f"\n  Half-life vs log10(c) regression:")
    log_cs = []
    hls = []
    for c in C_VALUES:
        if c > 1:
            hl = np.nanmean(results[c]['half_lives'])
            if np.isfinite(hl):
                log_cs.append(np.log10(c))
                hls.append(hl)
                print(f"    log10({c}) = {np.log10(c):.1f}  -->  half-life = {hl:.1f}")

    if len(log_cs) > 2:
        coeffs = np.polyfit(log_cs, hls, 1)
        slope = coeffs[0]
        intercept = coeffs[1]
        # R-squared
        predicted = np.polyval(coeffs, log_cs)
        ss_res = np.sum((np.array(hls) - predicted)**2)
        ss_tot = np.sum((np.array(hls) - np.mean(hls))**2)
        r_sq = 1 - ss_res / max(ss_tot, 1e-30)
        print(f"  Linear fit: half-life = {slope:.1f} * log10(c) + {intercept:.1f}")
        print(f"  R-squared = {r_sq:.3f}")
        print(f"  Interpretation: each 10x increase in c adds ~{slope:.0f} correction steps")
    else:
        slope = float('nan')

    # Hypothesis tests
    print(f"\n\n{'=' * 100}")
    print("HYPOTHESIS TESTS")
    print(f"{'=' * 100}")

    t1 = np.isfinite(slope) and slope < 50  # Less than 50 steps per decade
    print(f"\n  T1: Half-life scales as O(log(c)) (<50 steps/decade)?")
    print(f"      Slope = {slope:.1f} steps/decade")
    print(f"      --> {'PASS' if t1 else 'FAIL'}")
    if t1:
        print(f"      Interpretation: Self-correction is logarithmically efficient. Each order")
        print(f"      of magnitude increase in rescaling adds only ~{slope:.0f} extra steps to")
        print(f"      reach half the initial imbalance. This is consistent with exponential")
        print(f"      decay of the irrelevant gauge perturbation.")
    else:
        print(f"      Interpretation: Self-correction is slower than logarithmic, suggesting")
        print(f"      the NS normalization alone is insufficient for extreme rescalings.")

    t2 = all(np.nanmean(results[c]['half_lives']) < 50
             for c in C_VALUES if c <= 1000
             and np.isfinite(np.nanmean(results[c]['half_lives'])))
    print(f"\n  T2: Half-life < 50 for all c <= 1000?")
    for c in [cc for cc in C_VALUES if cc <= 1000]:
        hl = np.nanmean(results[c]['half_lives'])
        status = "OK" if hl < 50 else "TOO SLOW"
        print(f"      c={c}: {hl:.1f} steps [{status}]")
    print(f"      --> {'PASS' if t2 else 'FAIL'}")
    if t2:
        print(f"      Interpretation: In the practical regime (c <= 1000), Muon self-corrects")
        print(f"      within the first 50 steps -- well within a typical warm-up phase.")

    t3_checks = []
    for c in C_VALUES:
        hl = np.nanmean(results[c]['half_lives'])
        onset = np.nanmean(results[c]['onsets'])
        if np.isfinite(hl) and np.isfinite(onset):
            t3_checks.append(onset < hl)
    t3 = any(t3_checks) if t3_checks else False
    print(f"\n  T3: Training onset before half-life (learns while rebalancing)?")
    for c in C_VALUES:
        hl = np.nanmean(results[c]['half_lives'])
        onset = np.nanmean(results[c]['onsets'])
        if np.isfinite(hl) and np.isfinite(onset):
            print(f"      c={c}: onset={onset:.0f}, half-life={hl:.0f} --> {'onset FIRST' if onset < hl else 'half-life FIRST'}")
    print(f"      --> {'PASS' if t3 else 'FAIL'}")
    if t3:
        print(f"      Interpretation: Muon begins productive optimization BEFORE the momentum")
        print(f"      imbalance is fully corrected. The gauge-fixing and loss optimization")
        print(f"      proceed concurrently, not sequentially.")

    # Summary
    print(f"\n\n{'=' * 100}")
    print("SUMMARY INTERPRETATION")
    print(f"{'=' * 100}")
    n_pass = sum([t1, t2, t3])
    print(f"  Tests passed: {n_pass}/3")
    if n_pass == 3:
        print(f"  STRONG SUPPORT: Muon's self-correction is fast (O(log c)), practical")
        print(f"  (< 50 steps for c <= 1000), and non-blocking (learning starts before")
        print(f"  correction completes). This validates the RG picture where gauge")
        print(f"  perturbations decay exponentially under the flow.")
    elif n_pass >= 2:
        print(f"  MODERATE SUPPORT: Most aspects of the self-correction mechanism work")
        print(f"  as predicted. Check which test failed for potential limitations.")
    else:
        print(f"  WEAK/NO SUPPORT: The self-correction mechanism may be less efficient")
        print(f"  than the RG picture predicts, or the timescales may depend on factors")
        print(f"  beyond simple logarithmic scaling.")

    print(f"\n{'=' * 100}")
    print("EXPERIMENT COMPLETE")
    print(f"{'=' * 100}")

In [ ]:
if __name__ == '__main__':
    main()

## Conclusions

### What This Experiment Reveals

This experiment quantifies a **key operational property** of Muon: its ability to
self-correct from extreme initial scale imbalances. The half-life measurement provides
a concrete, reproducible number for how quickly the Newton-Schulz normalization
absorbs gauge perturbations.

### Interpretation Guide

- **All tests pass**: Muon's self-correction is remarkably efficient. The logarithmic
  scaling means that even $10^4$x rescaling only adds a modest number of correction
  steps. The concurrent learning/correction finding suggests Muon doesn't "waste"
  early training steps on rebalancing -- it optimizes the loss simultaneously.

- **T1 fails (scaling too steep)**: The correction mechanism degrades for very large $c$.
  This would suggest a practical upper bound on the rescaling Muon can handle efficiently.

- **T3 fails (onset after half-life)**: Learning is blocked until correction completes.
  This would mean Muon's robustness comes at a cost of delayed optimization.

### Connection to the Broader Framework

- **Exp 2.2b**: Established that Muon survives extreme rescaling (this experiment quantifies *how*)
- **Exp 2.2a**: Tested whether rescaling changes the endpoint (it shouldn't under gauge invariance)
- **H18a/b**: LR stability mechanism -- related question of what keeps Muon stable
- **RG interpretation**: The half-life is the relaxation time of an irrelevant operator
  under the RG flow. Fast relaxation (small half-life) means the gauge-fixing fixed
  point has a large spectral gap, indicating robust stability.